# 03 · Same-cell controls, conformal intervals, deployability

Each model uses the configuration its inner loop chose most often in notebook 02 (that
choice never saw a held-out cell). Answers R1-3, R1-5 and R1-9.

In [1]:
%run ./00_common.ipynb

root: <repository root>


In [2]:
cells = load_cells()
scr = screen(cells, save=False)
elig = scr[scr["eligible"]]["battery"].tolist()
df, cols = feature_table(cells, elig, "trailing")
df["age"] = df["cycle"].astype(float)
ch = pd.read_csv(RESULTS / "nested_choices.csv")
cfg = ch.groupby("model")["chosen"].agg(lambda s: s.value_counts().idxmax()).to_dict()
pd.Series(cfg, name="config").to_csv(RESULTS / "final_config.csv")
fp = lambda name, tr, te: fit_predict(name, tr, te, cols, *CONFIGS[cfg[name]])
models = list(get_models().keys())
cfg

{'LightGBM': 'base',
 'Linear': 'base',
 'RandomForest': 'age_ratio',
 'Ridge': 'base',
 'XGBoost': 'age_ratio'}

## Same-cell controls (R1-3)
A: own first 60% only (submitted control). B: nine other cells + own 60% (matched).
C: nine cells only (LOBO). All scored on the same last 40% of each cell. lambda = 0.5
falls inside the training range and is never scored.

In [3]:
sc = {"A_own60": {}, "B_9plus60": {}, "C_9only": {}}
for b in elig:
    g = df[df.battery_id == b].sort_values("cycle")
    n = int(len(g) * 0.6)
    own, te = g.iloc[:n], g.iloc[n:]
    others = df[df.battery_id != b]
    for tag, tr in (("A_own60", own), ("B_9plus60", pd.concat([others, own])), ("C_9only", others)):
        for m in models:
            sc[tag].setdefault(m, {})[b] = dict(cycle=te["cycle"].to_numpy(), rul_pred=fp(m, tr, te),
                                               rul_true=te["rul"].to_numpy(float))
rows = []
for tag, P in sc.items():
    e = per_cell_errors(P, cells)
    for (m, l), g in e.groupby(["method", "lambda"]):
        rows.append(dict(control=tag, method=m, **{"lambda": l}, n_cells=len(g), MAE=g["abs_err"].mean()))
    rows.append(dict(control=tag, method="(any)", **{"lambda": 0.5}, n_cells=int((e["lambda"] == 0.5).sum()), MAE=np.nan))
scdf = pd.DataFrame(rows)
scdf.to_csv(RESULTS / "final_same_cell.csv", index=False)
scdf.pivot_table(index="method", columns=["control", "lambda"], values="MAE").round(2)

control      A_own60        B_9plus60        C_9only       
lambda           0.7    0.9       0.7    0.9     0.7    0.9
method                                                     
LightGBM       16.59  34.85      7.19   9.59   13.06   5.28
Linear          4.47  13.80     16.69  19.77   21.23  15.26
RandomForest   12.10  41.91      6.01  12.12    9.36   4.13
Ridge           3.09   6.52     11.00  24.12   12.36  16.95
XGBoost        11.57  36.33      6.12  11.77    9.55   8.33

## Conformal intervals (R1-5)

In [4]:
t0 = time.time()
cp, bands = [], []
for held in elig:
    train, test = df[df.battery_id != held], df[df.battery_id == held].sort_values("cycle")
    for m in models:
        r, bnd = conformal_fold(m, train, test, fp, SEED, held, cells[held]["eol80"])
        cp += r
        if m == "XGBoost":
            bands += bnd
    print(f"conformal {held} done ({time.time() - t0:.0f}s)")
cpd = pd.DataFrame(cp)
cpd.to_csv(RESULTS / "final_conformal.csv", index=False)
pd.concat(bands).drop(columns="model").to_csv(RESULTS / "final_conformal_bands_xgb.csv", index=False)
cpd.groupby(["method", "model"])[["coverage", "cov_l0.5", "cov_l0.7", "cov_l0.9", "halfwidth", "width_over_life"]].mean().round(3)

conformal B0005 done (18s)


conformal B0006 done (37s)


conformal B0007 done (57s)


conformal B0018 done (79s)


conformal B0036 done (101s)


conformal B0042 done (127s)


conformal B0043 done (148s)


conformal B0044 done (169s)


conformal B0054 done (190s)


conformal B0055 done (210s)


coverage  cov_l0.5  cov_l0.7  cov_l0.9  halfwidth  \
method model                                                             
cvplus LightGBM         0.918       0.9       0.9       1.0     85.432   
       Linear           0.912       0.9       0.9       1.0    103.284   
       RandomForest     0.918       0.9       0.9       1.0     80.488   
       Ridge            0.927       0.9       0.9       1.0     70.570   
       XGBoost          0.931       1.0       0.9       1.0     77.840   
split2 LightGBM         0.821       0.8       0.9       1.0     38.985   
       Linear           0.752       0.7       0.8       0.8     57.473   
       RandomForest     0.829       0.9       0.9       1.0     41.968   
       Ridge            0.821       0.7       0.9       0.7     41.336   
       XGBoost          0.809       0.8       1.0       0.9     44.169   

                     width_over_life  
method model                          
cvplus LightGBM                1.263  
       Linear                  1.532  
       RandomForest            1.189  
       Ridge                   1.028  
       XGBoost                 1.138  
split2 LightGBM                0.533  
       Linear                  0.831  
       RandomForest            0.591  
       Ridge                   0.611  
       XGBoost                 0.646

In [5]:
cpd[cpd.model == "XGBoost"][["method", "held_out", "coverage", "width_over_life"]].round(2)

,method,held_out,coverage,width_over_life
6,split2,B0005,0.88,0.33
7,cvplus,B0005,1.00,0.83
16,split2,B0006,0.37,0.33
17,cvplus,B0006,1.00,1.33
26,split2,B0007,0.82,0.24
27,cvplus,B0007,1.00,0.75
36,split2,B0018,1.00,1.49
37,cvplus,B0018,1.00,1.11
46,split2,B0036,0.23,0.08
47,cvplus,B0036,0.31,0.15


## Deployability (R1-9)
Serialized size, compact C-array size (internal node 9 B, leaf 4 B), inference RAM for
trees kept in flash, desktop latency (median of 3 runs).

In [6]:
dep = []
for m in models:
    age, ratio = CONFIGS[cfg[m]]
    cc = cols + (["age"] if age else [])
    X, y, _ = xy(df, cc)
    mod = get_models()[m]
    mod.fit(X, y / df["cycle"].to_numpy(float) if ratio else y)
    buf = io.BytesIO(); pickle.dump(mod, buf)
    lat = []
    for _ in range(3):
        t = time.perf_counter()
        for _ in range(20):
            mod.predict(X[:200])
        lat.append((time.perf_counter() - t) / 4000 * 1e6)
    nodes = leaves = ntree = depth = None
    if m == "RandomForest":
        ntree = len(mod.estimators_); depth = max(e.tree_.max_depth for e in mod.estimators_)
        nodes = sum(e.tree_.node_count for e in mod.estimators_); leaves = sum(e.tree_.n_leaves for e in mod.estimators_)
    elif m == "XGBoost":
        t = mod.get_booster().trees_to_dataframe(); ntree = t["Tree"].nunique(); nodes = len(t)
        leaves = int((t["Feature"] == "Leaf").sum()); depth = 4
    elif m == "LightGBM":
        t = mod.booster_.trees_to_dataframe(); ntree = t["tree_index"].nunique(); nodes = len(t)
        leaves = int(t["split_feature"].isna().sum()); depth = int(t["node_depth"].max() - 1)
    dep.append(dict(model=m, config=cfg[m], n_features=len(cc), n_trees=ntree, max_depth=depth, n_nodes=nodes,
                    pickle_kb=round(buf.getbuffer().nbytes / 1024, 1),
                    flash_c_export_kb=None if nodes is None else round(((nodes - leaves) * 9 + leaves * 4) / 1024, 1),
                    ram_inference_bytes=len(cc) * 4 + 4 + 64, latency_us=round(sorted(lat)[1], 2)))
dd = pd.DataFrame(dep)
dd.to_csv(RESULTS / "final_deploy.csv", index=False)
dd

,model,config,n_features,n_trees,max_depth,n_nodes,pickle_kb,flash_c_export_kb,ram_inference_bytes,latency_us
0,Linear,base,19,NaN,NaN,NaN,0.8,NaN,144,0.72
1,Ridge,base,19,NaN,NaN,NaN,0.6,NaN,144,0.50
2,RandomForest,age_ratio,20,300.0,25.0,270036.0,19061.9,1713.4,148,408.45
3,XGBoost,age_ratio,20,400.0,4.0,9888.0,595.6,61.8,148,5.51
4,LightGBM,base,19,400.0,4.0,5974.0,328.1,36.9,144,9.21
